### Classification 
{SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED}.


-[train-claims,dev-claims].json: JSON files for the labelled training and development set;

-[test-claims-unlabelled].json: JSON file for the unlabelled test set;

-evidence.json: JSON file containing a large number of evidence passages (i.e. the “knowledge source”);

-dev-claims-baseline.json: JSON file containing predictions of a baseline system on the development set;
-eval.py: Python script to evaluate system performance (see “Evaluation” below for more details).

In [1]:
import json
import numpy as np
import re

import time

import time
from datetime import datetime
import os
import pandas as pd


from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [2]:
dev_baseline = 'data\dev-claims-baseline.json'
dev_claims = 'data\dev-claims.json'

train_claims_path = 'data/train-claims.json'
test_claims_path = 'data/test-claims-unlabelled.json'

evidence_path = 'data\evidence.json'


### Read and storage Data

In [3]:
# For labelled data
def load_json_to_dataframe(json_file):
    # Load the JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Create empty lists to store the data
    ids = []
    claim_texts = []
    claim_labels = []
    evidences = []

    # Iterate over each claim in the JSON data
    for claim_id, claim_data in data.items():
        # Extract claim details
        ids.append(claim_id.split('-')[1])
        claim_texts.append(claim_data["claim_text"])
        claim_labels.append(claim_data["claim_label"])
        evidences.append(claim_data["evidences"])

    # Create a DataFrame
    df = pd.DataFrame({
        'id': ids,
        'claim_text': claim_texts,
        'claim_label': claim_labels,
        'evidences': evidences
    })

    return df


def load_evidnece(json_file):
    # 读取 JSON 文件
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 创建一个空的 DataFrame
    df = pd.DataFrame(data.items(), columns=['id', 'text'])
    
    return df

def load_test(json_file):
    # Load the JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Create empty lists to store the data
    ids = []
    claim_texts = []
    
    # Iterate over each claim in the JSON data
    for claim_id, claim_data in data.items():
        # Extract claim details
        ids.append(claim_id.split('-')[1])
        claim_texts.append(claim_data["claim_text"])
        

    # Create a DataFrame
    df = pd.DataFrame({
        'id': ids,
        'claim_text': claim_texts,
    })

    return df



In [4]:
# 检查有没有少读
def count_claim_entries(json_file):
    # 读取 JSON 文件
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 初始化计数器
    count = 0

    # 遍历 JSON 数据，计算具有指定 "claim_id" 的条目数
    for key in data:
        
        count += 1

    return count

# 调用函数并打印结果
count = count_claim_entries(test_claims_path)
print(f"The number of entries is: {count}")


The number of entries is: 153


In [5]:
df_test = load_test(test_claims_path)
df_test 

,id,claim_text
0,2967,The contribution of waste heat to the global c...
1,979,“Warm weather worsened the most recent five-ye...
2,1609,Greenland has only lost a tiny fraction of its...
3,1020,“The global reef crisis does not necessarily m...
4,2599,Small amounts of very active substances can ca...
...,...,...
148,293,When the measuring equipment gets old and need...
149,910,"The cement, iron and steel, and petroleum refi..."
150,2815,A new peer-reviewed study on Surface Warming a...
151,1652,The strong CO2 effect has been observed by man...


In [6]:
df_evidence = load_evidnece(evidence_path)
row1 = df_evidence.loc[df_evidence['id'] == 'evidence-67732']
row1.iloc[0, 1]


'[citation needed] South Australia has the highest retail price for electricity in the country.'

In [7]:
row2 = df_evidence.loc[df_evidence['id'] == 'evidence-572512']
row2.iloc[0, 1]

'"South Australia has the highest power prices in the world".'

In [8]:
df_dv = load_json_to_dataframe(dev_claims)
df_dv.head(1)

,id,claim_text,claim_label,evidences
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]"


In [9]:
df_dv.head(1)

,id,claim_text,claim_label,evidences
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]"


In [10]:
df_train = load_json_to_dataframe('data/train-claims.json')
df_train.head(5)
    

,id,claim_text,claim_label,evidences
0,1937,Not only is there no scientific evidence that ...,DISPUTED,"[evidence-442946, evidence-1194317, evidence-1..."
1,126,El Niño drove record highs in global temperatu...,REFUTES,"[evidence-338219, evidence-1127398]"
2,2510,"In 1946, PDO switched to a cool phase.",SUPPORTS,"[evidence-530063, evidence-984887]"
3,2021,Weather Channel co-founder John Coleman provid...,DISPUTED,"[evidence-1177431, evidence-782448, evidence-5..."
4,2449,"""January 2008 capped a 12 month period of glob...",NOT_ENOUGH_INFO,"[evidence-1010750, evidence-91661, evidence-72..."


In [11]:
df_dv['evidence_texts'] = df_dv['evidences'].apply(
    lambda ids: [df_evidence[df_evidence['id'] == evidence]['text'].iloc[0] for evidence in ids]
)


In [12]:
df_dv.head(1)

,id,claim_text,claim_label,evidences,evidence_texts
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]",[[citation needed] South Australia has the hig...


In [3]:
pip install scikit-learn

   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --- ------------------------------------ 0.7/9.3 MB 14.6 MB/s eta 0:00:01
   ----- ---------------------------------- 1.3/9.3 MB 14.1 MB/s eta 0:00:01
   -------- ------------------------------- 2.0/9.3 MB 13.8 MB/s eta 0:00:01
   ----------- ---------------------------- 2.6/9.3 MB 13.7 MB/s eta 0:00:01
   ------------- -------------------------- 3.2/9.3 MB 13.6 MB/s eta 0:00:01
   ---------------- ----------------------- 3.8/9.3 MB 13.5 MB/s eta 0:00:01
   ------------------- -------------------- 4.5/9.3 MB 13.6 MB/s eta 0:00:01
   --------------------- ------------------ 5.0/9.3 MB 13.4 MB/s eta 0:00:01
   ------------------------ --------------- 5.7/9.3 MB 13.5 MB/s eta 0:00:01
   --------------------------- ------------ 6.3/9.3 MB 13.9 MB/s eta 0:00:01
   ---------------------------- ----------- 6.7/9.3 MB 13.7 MB/s eta 0:00:01
   ------------------------------ --------- 7.1/9.3 MB 12.6 MB/s eta 0:00:01
   ----

### 不能去掉stop word， 应为要判断支不支持

### Build models

In [14]:

from sklearn.feature_extraction.text import CountVectorizer



# 假设df是你的DataFrame，包含claim_text, claim_label和evidence_texts列

# 将claim_text和evidence_texts列合并为一列，作为模型的输入文本
X = df_dv['claim_text'] + df_dv['evidence_texts'].apply(lambda x: ' '.join(x))
y = df_dv['claim_label']

# 划分数据集为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 使用CountVectorizer向量化文本
count_vectorizer = CountVectorizer()

# 在训练集上拟合CountVectorizer，并转换训练集和测试集
X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

# 定义要尝试的C值
C_values = [0.01, 0.1, 1.0, 10.0]

# 创建空列表来存储每个C值的准确率
accuracy_scores = []

# 在不同的C值下训练模型并评估性能
for C in C_values:
    # 训练逻辑回归分类器
    classifier = LogisticRegression(max_iter=1000, C=C)
    classifier.fit(X_train_count, y_train)
    
    # 预测测试集
    y_pred = classifier.predict(X_test_count)
    
    # 计算准确率并添加到列表中
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_scores.append(accuracy)
    
    # 打印当前C值的准确率
    print(f"C = {C}: Accuracy = {accuracy}")

# 打印不同C值下的准确率
print("Accuracy scores:", accuracy_scores)


C = 0.01: Accuracy = 0.45161290322580644
C = 0.1: Accuracy = 0.4838709677419355
C = 1.0: Accuracy = 0.4838709677419355
C = 10.0: Accuracy = 0.41935483870967744
Accuracy scores: [0.45161290322580644, 0.4838709677419355, 0.4838709677419355, 0.41935483870967744]


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 定义要尝试的参数组合
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

# 创建空列表来存储每个参数组合的准确率
accuracy_scores_rf = []

# 在不同的参数组合下训练模型并评估性能
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        # 创建随机森林分类器
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        
        # 在训练集上拟合模型
        rf_classifier.fit(X_train_count, y_train)
        
        # 在测试集上进行预测
        y_pred_rf = rf_classifier.predict(X_test_count)
        
        # 计算准确率并添加到列表中
        accuracy_rf = accuracy_score(y_test, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        
        # 打印当前参数组合的准确率
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

# 打印不同参数组合下的准确率
print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")


n_estimators = 50, max_depth = None: Accuracy = 0.2903225806451613
n_estimators = 50, max_depth = 10: Accuracy = 0.3225806451612903
n_estimators = 50, max_depth = 20: Accuracy = 0.2903225806451613
n_estimators = 100, max_depth = None: Accuracy = 0.3225806451612903
n_estimators = 100, max_depth = 10: Accuracy = 0.3225806451612903
n_estimators = 100, max_depth = 20: Accuracy = 0.3225806451612903
n_estimators = 200, max_depth = None: Accuracy = 0.3225806451612903
n_estimators = 200, max_depth = 10: Accuracy = 0.3225806451612903
n_estimators = 200, max_depth = 20: Accuracy = 0.3548387096774194
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.2903225806451613
Parameters: (50, 10), Accuracy: 0.3225806451612903
Parameters: (50, 20), Accuracy: 0.2903225806451613
Parameters: (100, None), Accuracy: 0.3225806451612903
Parameters: (100, 10), Accuracy: 0.3225806451612903
Parameters: (100, 20), Accuracy: 0.3225806451612903
Parameters: (200, None), Accuracy: 0.3225806451612903
P

In [16]:
from sklearn.svm import SVC


# 定义要尝试的参数组合
C_values = [0.01, 0.1, 1.0, 10.0]
kernel_values = ['linear', 'rbf']

# 创建空列表来存储每个参数组合的准确率
accuracy_scores_svm = []

# 在不同的参数组合下训练模型并评估性能
for C in C_values:
    for kernel in kernel_values:
        # 创建SVM分类器
        svm_classifier = SVC(C=C, kernel=kernel, random_state=42)
        
        # 在训练集上拟合模型
        svm_classifier.fit(X_train_count, y_train)
        
        # 在测试集上进行预测
        y_pred_svm = svm_classifier.predict(X_test_count)
        
        # 计算准确率并添加到列表中
        accuracy_svm = accuracy_score(y_test, y_pred_svm)
        accuracy_scores_svm.append(((C, kernel), accuracy_svm))
        
        # 打印当前参数组合的准确率
        print(f"C = {C}, kernel = {kernel}: Accuracy = {accuracy_svm}")

# 打印不同参数组合下的准确率
print("Accuracy scores for SVM:")
for params, accuracy in accuracy_scores_svm:
    print(f"Parameters: {params}, Accuracy: {accuracy}")


C = 0.01, kernel = linear: Accuracy = 0.45161290322580644
C = 0.01, kernel = rbf: Accuracy = 0.2903225806451613
C = 0.1, kernel = linear: Accuracy = 0.45161290322580644
C = 0.1, kernel = rbf: Accuracy = 0.2903225806451613
C = 1.0, kernel = linear: Accuracy = 0.45161290322580644
C = 1.0, kernel = rbf: Accuracy = 0.45161290322580644
C = 10.0, kernel = linear: Accuracy = 0.45161290322580644
C = 10.0, kernel = rbf: Accuracy = 0.4838709677419355
Accuracy scores for SVM:
Parameters: (0.01, 'linear'), Accuracy: 0.45161290322580644
Parameters: (0.01, 'rbf'), Accuracy: 0.2903225806451613
Parameters: (0.1, 'linear'), Accuracy: 0.45161290322580644
Parameters: (0.1, 'rbf'), Accuracy: 0.2903225806451613
Parameters: (1.0, 'linear'), Accuracy: 0.45161290322580644
Parameters: (1.0, 'rbf'), Accuracy: 0.45161290322580644
Parameters: (10.0, 'linear'), Accuracy: 0.45161290322580644
Parameters: (10.0, 'rbf'), Accuracy: 0.4838709677419355


In [17]:
df_train['evidence_texts'] = df_train['evidences'].apply(
    lambda ids: [df_evidence[df_evidence['id'] == evidence]['text'].iloc[0] for evidence in ids]
)

X_training = df_train['claim_text'] + df_train['evidence_texts'].apply(lambda x: ' '.join(x))
y_training = df_train['claim_label']



In [18]:
# X_train_count2 = count_vectorizer.fit_transform(X_training)
X_train_count2 = count_vectorizer.transform(X_training)



### Apply on training set

In [19]:

# 训练逻辑回归分类器
classifier2 = LogisticRegression(max_iter=1000, C=0.1)
classifier2.fit(X_train_count, y_train)
    
# 预测测试集
y_pred2 = classifier2.predict(X_train_count2)
    
    # 计算准确率并添加到列表中
accuracy = accuracy_score(y_training , y_pred2)

# 打印不同C值下的准确率
print("Accuracy scores:", accuracy)






Accuracy scores: 0.5252442996742671


### LSTM

In [20]:
df_dv['evidence_texts']

0      [[citation needed] South Australia has the hig...
1      [The 2011 UNEP Green Economy report states tha...
2      [Multiple independently produced instrumental ...
3      [Genetic disorders are the result of deleterio...
4      [If iceberg calving has happened as an average...
                             ...                        
149    [As is stated in Article 2 of the Convention, ...
150    [Increases in atmospheric concentrations of CO...
151    [Tropical waters contain few nutrients yet a c...
152    [A 2007 study by David Douglass and coworkers,...
153    [The poleward migration of coral species refer...
Name: evidence_texts, Length: 154, dtype: object

In [21]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np
from scipy.spatial.distance import euclidean
from scipy.optimize import linear_sum_assignment

In [22]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import download
import pandas as pd


In [23]:
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    return [word for word in tokens if word.isalnum() and word not in stop_words]

In [24]:
df_evidence.iloc[0, 1]

'John Bennet Lawes, English entrepreneur and agricultural scientist'

In [25]:
preprocess_text('John Bennet Lawes, English entrepreneur and 2 agricultural scientist')

['john',
 'bennet',
 'lawes',
 'english',
 'entrepreneur',
 '2',
 'agricultural',
 'scientist']

In [26]:
df_evidence['preprocessed_text'] = df_evidence['text'].apply(preprocess_text)

#### Pipeline
- 训练unsupervised的模型
- 利用trained的语言模型去计算wmd
- 利用wmd选evidence

In [27]:



# 预处理 col2 中的列表
def preprocess_list(lst):
    return [preprocess_text(text) for text in lst]

# 训练 Word2Vec 模型
word2vec_model = Word2Vec(vector_size=100, window=5, min_count=1, workers=4)

# 构建初始的词汇表
all_tokens = [preprocess_text(claim) for claim in df_dv['claim_text']]
all_tokens += [preprocess_text(text) for sublist in df_dv['evidence_texts'] for text in sublist]

# 扁平化列表
all_tokens = [token for sublist in all_tokens for token in sublist]

# 构建词汇表
word2vec_model.build_vocab(all_tokens)

# 训练模型
for claim, evidences in zip(df_dv['claim_text'], df_dv['evidence_texts']):
    claim_tokens = preprocess_text(claim)
    evidence_tokens = preprocess_list(evidences)
    # 将声明和证据都传递给模型进行训练
    word2vec_model.train([claim_tokens, *evidence_tokens], total_examples=word2vec_model.corpus_count, epochs=5)

    

In [35]:


def wmdistance(claim_tokens, evidence_tokens, model):
    # 计算文档之间的距离矩阵
    distance_matrix = np.zeros((len(claim_tokens), len(evidence_tokens)))
    for i, vec1 in enumerate(claim_tokens):
        for j, vec2 in enumerate(evidence_tokens):
            distance_matrix[i, j] = euclidean(vec1, vec2)
    
    # 使用线性优化算法找到最佳单词对齐
    row_ind, col_ind = linear_sum_assignment(distance_matrix)
    
    # 计算WMD
    total_cost = distance_matrix[row_ind, col_ind].sum()
    
    return total_cost

def find_most_related_evidence(claim, evidence_df, model):
    # 初始化最小距离和对应的证据 ID
    max_distance = 0.177
    related_evidence_id = []
    min_distance = float('inf')
    most_related_evidence_id = None
    # 移除停用词
    stop_words = set(stopwords.words('english'))
    
    claim_tokens = [model.wv[word] for word in word_tokenize(claim.lower()) if word not in stop_words and word in model.wv]
    
    # 计算声明与每个证据文本之间的 WMD
    for index, row in evidence_df.iterrows():
        evidence_id = row['id']
        evidence_text = row['text']
        
        # 
        evidence_tokens = [model.wv[word] for word in word_tokenize(evidence_text.lower()) if word not in stop_words and word in model.wv]
        
        
        # 计算 WMD
        distance = wmdistance(claim_tokens, evidence_tokens, model)
        
        # 更新最相关的证据 ID
        # 是个问题， 多少相似度会是相关呢
        if distance < max_distance and  distance < min_distance:
            min_distance = distance
            most_related_evidence_id = evidence_id
            related_evidence_id.append(evidence_id)
    
    return most_related_evidence_id 



In [37]:
df_dv

,id,claim_text,claim_label,evidences,evidence_texts
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]",[[citation needed] South Australia has the hig...
1,375,when 3 per cent of total annual global emissio...,NOT_ENOUGH_INFO,"[evidence-996421, evidence-1080858, evidence-2...",[The 2011 UNEP Green Economy report states tha...
2,1266,This means that the world is now 1C warmer tha...,SUPPORTS,"[evidence-889933, evidence-694262]",[Multiple independently produced instrumental ...
3,871,"“As it happens, Zika may also be a good model ...",NOT_ENOUGH_INFO,"[evidence-422399, evidence-702226, evidence-28...",[Genetic disorders are the result of deleterio...
4,2164,Greenland has only lost a tiny fraction of its...,REFUTES,"[evidence-52981, evidence-264761, evidence-947...",[If iceberg calving has happened as an average...
...,...,...,...,...,...
149,2400,"'To suddenly label CO2 as a ""pollutant"" is a d...",REFUTES,"[evidence-409365, evidence-127519, evidence-85...","[As is stated in Article 2 of the Convention, ..."
150,204,"after a natural orbitally driven warming, atmo...",NOT_ENOUGH_INFO,"[evidence-368192, evidence-261690, evidence-20...",[Increases in atmospheric concentrations of CO...
151,1426,Many of the world’s coral reefs are already ba...,NOT_ENOUGH_INFO,"[evidence-1124018, evidence-995813, evidence-1...",[Tropical waters contain few nutrients yet a c...
152,698,A recent study led by Lawrence Livermore Natio...,REFUTES,[evidence-660755],"[A 2007 study by David Douglass and coworkers,..."


In [34]:
df_evidence

,id,text,preprocessed_text
0,evidence-0,"John Bennet Lawes, English entrepreneur and ag...","[john, bennet, lawes, english, entrepreneur, a..."
1,evidence-1,Lindberg began his professional career at the ...,"[lindberg, began, professional, career, age, 1..."
2,evidence-2,``Boston (Ladies of Cambridge)'' by Vampire We...,"[boston, ladies, cambridge, vampire, weekend]"
3,evidence-3,"Gerald Francis Goyer (born October 20, 1936) w...","[gerald, francis, goyer, born, october, 20, 19..."
4,evidence-4,He detected abnormalities of oxytocinergic fun...,"[detected, abnormalities, oxytocinergic, funct..."
...,...,...,...
1208822,evidence-1208822,Also on the property is a contributing garage ...,"[also, property, contributing, garage, apartment]"
1208823,evidence-1208823,| class = ``fn org'' | Fyrde | | | | 6110 | | ...,"[class, fn, org, fyrde, 6110, volda]"
1208824,evidence-1208824,"Dragon Storm (game), a role-playing game and c...","[dragon, storm, game, game, collectible, card,..."
1208825,evidence-1208825,It states that the Zeriuani ``which is so grea...,"[states, zeriuani, great, realm, tradition, re..."


In [36]:
find_most_related_evidence(df_dv.iloc[0, 1], df_evidence, word2vec_model)

'evidence-0'

In [32]:
for claim in df_dv['claim_text']:
    find_most_related_evidence(claim, df_evidence, word2vec_model)

KeyboardInterrupt: 

#### 找到最大的wmd

In [ ]:
def wmdistance2(claim_tokens, evidence_tokens, model):
    # 计算文档之间的距离矩阵
    distance_matrix = np.zeros((len(claim_tokens), len(evidence_tokens)))
    for i, vec1 in enumerate(claim_tokens):
        for j, vec2 in enumerate(evidence_tokens):
            distance_matrix[i, j] = euclidean(vec1, vec2)
    
    # 使用线性优化算法找到最佳单词对齐
    row_ind, col_ind = linear_sum_assignment(distance_matrix)
    
    # 计算WMD
    total_cost = distance_matrix[row_ind, col_ind].sum()
    
    return total_cost

def find_max_wmd2(df, model):
    max_wmd = -1  # 初始化最大的 WMD 值
    max_wmd_evidence = None  # 初始化最大 WMD 对应的证据文本
    
    # 移除停用词
    stop_words = set(stopwords.words('english'))
    
    for index, row in df.iterrows():
        claim = row['claim_text']
        evidence_sentences = row['evidence_texts']  # 获取证据句子列表
        
        # 遍历证据句子
        for evidence_sentence in evidence_sentences:
            # 分词并移除停用词
            claim_tokens = [model.wv[word] for word in word_tokenize(claim.lower()) if word not in stop_words and word in model.wv]
            evidence_tokens = [model.wv[word] for word in word_tokenize(evidence_sentence.lower()) if word not in stop_words and word in model.wv]
            
            # 如果没有有效的词向量，跳过当前句子
            if not claim_tokens or not evidence_tokens:
                continue
            
            # 计算 WMD
            wmd = wmdistance2(claim_tokens, evidence_tokens, model)
            
            # 更新最大 WMD 值和对应的证据文本
            if wmd > max_wmd:
                max_wmd = wmd
                max_wmd_evidence = evidence_sentence
    
    return max_wmd, max_wmd_evidence


In [ ]:
df_dv

,id,claim_text,claim_label,evidences,evidence_texts
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]",[[citation needed] South Australia has the hig...
1,375,when 3 per cent of total annual global emissio...,NOT_ENOUGH_INFO,"[evidence-996421, evidence-1080858, evidence-2...",[The 2011 UNEP Green Economy report states tha...
2,1266,This means that the world is now 1C warmer tha...,SUPPORTS,"[evidence-889933, evidence-694262]",[Multiple independently produced instrumental ...
3,871,"“As it happens, Zika may also be a good model ...",NOT_ENOUGH_INFO,"[evidence-422399, evidence-702226, evidence-28...",[Genetic disorders are the result of deleterio...
4,2164,Greenland has only lost a tiny fraction of its...,REFUTES,"[evidence-52981, evidence-264761, evidence-947...",[If iceberg calving has happened as an average...
...,...,...,...,...,...
149,2400,"'To suddenly label CO2 as a ""pollutant"" is a d...",REFUTES,"[evidence-409365, evidence-127519, evidence-85...","[As is stated in Article 2 of the Convention, ..."
150,204,"after a natural orbitally driven warming, atmo...",NOT_ENOUGH_INFO,"[evidence-368192, evidence-261690, evidence-20...",[Increases in atmospheric concentrations of CO...
151,1426,Many of the world’s coral reefs are already ba...,NOT_ENOUGH_INFO,"[evidence-1124018, evidence-995813, evidence-1...",[Tropical waters contain few nutrients yet a c...
152,698,A recent study led by Lawrence Livermore Natio...,REFUTES,[evidence-660755],"[A 2007 study by David Douglass and coworkers,..."


In [ ]:
find_max_wmd2(df_dv, word2vec_model)

(0.17696242034435272,
 'In the modern era, emissions to the atmosphere from volcanoes are approximately 0.645 billion tonnes of CO 2 per year, whereas humans contribute 29 billion tonnes of CO 2 each year.')